# Full pipeline

Neural Network-Driven Signal Generation for Systematic Portfolio Construction (MScFE 690 Capstone).

This notebook holds no logic of its own. Each step calls a module in `src/` and shows the result. Settings are in `config.yaml`.

In [ ]:
import os, sys

if "google.colab" in sys.modules and not os.path.exists("src"):
    !git clone https://github.com/Joebasshd/nn-etf-portfolio-signals.git repo
    %cd repo
    !pip install -q -r requirements.txt

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import data, evaluate, features, models, portfolio, signals, splits, stats

cfg = data.load_config(ROOT / "config.yaml")
RUN_LABEL = "src-full-quick" if cfg["run"]["quick_mode"] else "src-full"
clock = evaluate.start_clock()
cfg["run"]

## 1. Data

Yahoo prices and FRED macro series, cached in `data/raw`. In quick mode the prices are generated instead.

In [ ]:
if cfg["run"]["quick_mode"]:
    prices, macro = data.make_synthetic(cfg, n_days=3400)
else:
    prices, macro = data.load_data(cfg)

print(f"{len(prices)} trading days, {prices.index.min().date()} to {prices.index.max().date()}")
macro.dropna().tail()

In [ ]:
out = evaluate.results_dir(cfg, RUN_LABEL)
FIG = out / "figures"
print(f"Results: {out}")

## 2. Features and label

31 indicators per ETF. The label for a decision at the close of day t is the return from the close of t+1 to the close of t+6.

In [ ]:
dataset = features.build_dataset(prices, macro, cfg)
feature_cols = features.feature_columns(dataset)

print(f"{len(dataset):,} rows, {len(feature_cols)} features")
dataset[feature_cols].describe().T[["mean", "std", "min", "max"]].round(3)

In [ ]:
balance = dataset.groupby("ticker")["label"].mean().sort_values(ascending=False)

ax = balance.plot(kind="bar", color="steelblue", figsize=(11, 4), ylim=(0.4, 0.65))
ax.axhline(0.5, color="red", linestyle="--", linewidth=1, label="50/50")
ax.set_title("Share of positive 5-day labels by ETF")
ax.set_ylabel("Positive share")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Walk-forward splits

Training expands by one year at each step. Validation is the 12 months ending 11 trading days before the test year, and training ends 11 trading days before validation (1 execution lag + 5 horizon + 5 embargo), so each test year is complete.

In [ ]:
folds = splits.walk_forward_splits(prices.index, cfg)

pd.DataFrame(
    [
        {
            "test_year": s.test_year,
            "train_start": s.train.min().date(),
            "train_end": s.train.max().date(),
            "validation_start": s.validation.min().date(),
            "validation_end": s.validation.max().date(),
            "test_start": s.test.min().date(),
            "test_end": s.test.max().date(),
        }
        for s in folds
    ]
).set_index("test_year")

## 4. Models

In every window LASSO (L1) and LightGBM (top 15 by gain) only select features. The ridge (L2) logistic regression, LightGBM and the LSTM all forecast from the union of the two selections. The code key `lasso` is the ridge forecasting fit, labelled "Linear (L1 selection, ridge on union)". One LSTM per seed, plus the seed-averaged forecast.

In [ ]:
result = models.run_walk_forward(dataset, folds, cfg)
names = evaluate.display_names(result.probabilities.columns)
labels = dataset["label"]
print(f"{len(result.probabilities):,} test rows | models: {', '.join(names.values())}")
result.tuning

In [ ]:
tuning_edges = models.grid_edges(result.tuning, cfg)
tuning_edges

## 5. Signals

Main signal `s = min(1, max(0, (p − b) / (p90 − b)))`, with `b` from the training block and `p90` from validation forecasts. A model stays flat for a whole window when `p90 − b` is below the floor. The base-rate centred and `2p − 1` signals are sensitivity versions for the three main models.

In [ ]:
main_signals = signals.scaled_signal(result.probabilities, result.window_stats, cfg)
main_models = signals.main_models(cfg)
sensitivity = {
    "base_rate": signals.base_rate_centred_signal(result.probabilities[main_models], result.window_stats),
    "linear": signals.linear_signal(result.probabilities[main_models]),
}
edges = signals.signal_edges(result.window_stats, cfg).reset_index().assign(model=lambda e: e["model"].map(names))
flat = edges[edges["flat"]]
print(f"Flat windows: {len(flat)}")
print(flat.groupby("model")["test_year"].apply(list).to_string() if len(flat) else "none")
edges.pivot(index="test_year", columns="model", values="edge")[list(names.values())].round(4)

## 6. Portfolio and benchmarks

Inverse-volatility weights scaled to 10% forecast volatility, capped at 25% per ETF, no leverage, cash at the T-bill rate. Weekly decisions, traded at the next close, 2-point no-trade band per ETF, 5 bps per side. Every strategy and benchmark goes through the same backtest.

In [ ]:
market = portfolio.prepare_market(prices, macro, cfg, result.probabilities.index.get_level_values("date").min())
base = portfolio.base_weights(market, cfg)
strategies = evaluate.strategy_targets(main_signals, sensitivity, base, market, cfg, names)
results = portfolio.run_strategies(strategies, market, cfg)
common = next(iter(results.values())).index
performance = evaluate.performance_table(results, market.rf)
print(f"Backtest: {common[0].date()} to {common[-1].date()}, {len(common)} days")
performance.style.format({"Ann. return": "{:.2%}", "Ann. volatility": "{:.2%}", "Sharpe": "{:.2f}", "Sortino": "{:.2f}",
                          "Max drawdown": "{:.2%}", "Turnover / yr": "{:.1f}x", "Avg exposure": "{:.0%}", "Total cost": "{:.2%}"})

In [ ]:
shown = ["SPY buy-and-hold", "Vol-targeted SPY", "Always-on (vol-scaled)", names["lasso"], names["lightgbm"],
         names[main_models[2]]] + ([names["lstm_avg"]] if "lstm_avg" in names else [])
fig, ax = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for name in shown:
    equity = (1 + results[name]["ret"]).cumprod()
    ax[0].plot(equity, label=name)
    ax[1].plot(equity / equity.cummax() - 1, label=name)
ax[0].set_yscale("log")
ax[0].set_title(f"Growth of $1, net of {cfg['portfolio']['cost_bps_per_side']} bps per side (log scale)")
ax[0].legend(fontsize=8)
ax[1].set_title("Drawdown")
plt.tight_layout()
fig.savefig(FIG / "equity_drawdown.png", dpi=120)
plt.show()

In [ ]:
cost_names = ["Always-on (vol-scaled)", "SMA 50/200 rule"] + list(names.values())
costs = evaluate.cost_sensitivity({n: strategies[n] for n in cost_names}, market, cfg, common)
costs.round(2)

## 7. Prediction quality

Accuracy against the base rate, AUC, Brier score and log loss on every labelled test row. Calibration is analysis only: forecasts are binned by deciles of p.

In [ ]:
prediction = evaluate.prediction_metrics(result.probabilities, labels, names)
prediction.round(4)

In [ ]:
auc_year = evaluate.auc_by_year(result.probabilities, labels, names)
ax = auc_year[[names[m] for m in main_models] + ([names["lstm_avg"]] if "lstm_avg" in names else [])].plot(
    marker="o", figsize=(12, 4), title="Out-of-sample AUC by test year")
ax.axhline(0.5, color="grey", linestyle="--")
plt.tight_layout()
ax.figure.savefig(FIG / "auc_by_year.png", dpi=120)
plt.show()
auc_year.round(4)

In [ ]:
calibration = evaluate.calibration_table(result.probabilities, labels, names, cfg["stats"]["calibration_bins"])
fig, ax = plt.subplots(figsize=(6, 6))
for name in [names[m] for m in main_models] + ([names["lstm_avg"]] if "lstm_avg" in names else []):
    t = calibration[calibration["model"] == name]
    ax.plot(t["mean_p"], t["observed"], marker="o", label=name)
lims = [calibration[["mean_p", "observed"]].min().min(), calibration[["mean_p", "observed"]].max().max()]
ax.plot(lims, lims, color="grey", linestyle="--", label="perfect calibration")
ax.set_xlabel("Mean forecast (decile)")
ax.set_ylabel("Observed share of positive labels")
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIG / "calibration.png", dpi=120)
plt.show()

## 8. Indicator stability

How often each indicator is selected, the average Jaccard overlap between consecutive windows, and the share of selected indicators by family.

In [ ]:
frequency = evaluate.selection_frequency(result.selections, feature_cols)
families = evaluate.family_share(result.selections)
stability = evaluate.stability_summary(result.selections, feature_cols)
fig, ax = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [3, 2]})
frequency[["LASSO", "LightGBM"]].head(20).plot.barh(ax=ax[0], title="Share of windows in which each indicator is selected (top 20)")
ax[0].invert_yaxis()
families.plot.bar(ax=ax[1], rot=30, title="Average share of selected indicators by family")
plt.tight_layout()
fig.savefig(FIG / "selection_stability.png", dpi=120)
plt.show()
stability

## 9. Significance

**Sharpe differences.** Stationary block bootstrap, with the HAC Ledoit–Wolf (2008) test next to it. H4: every LSTM seed and the seed average against the linear model and LightGBM, Holm-corrected across the two. LSTM vs LightGBM is the substantive test; LSTM vs linear is reported but not interpreted, because the linear model finds no usable signal.

In [ ]:
sharpe_tests = stats.sharpe_tests(results, market.rf, names, cfg)
sharpe_tests.round(4)

**Exposure-matched placebo.** Each draw circularly shifts the model's actual weekly signals (independent: one shift per ETF; common: one shift for all ETFs) and runs them through the same backtest and costs. Exposure and holding lengths are kept; timing is not.

In [ ]:
placebo, placebo_draws = stats.run_placebo(main_signals, base, market, cfg, common, performance, names)
labels_shown = list(names.values())
variants = cfg["stats"]["placebo"]["variants"]
fig, axes = plt.subplots(1, len(variants), figsize=(7 * len(variants), 0.45 * len(labels_shown) + 1.5), sharey=True, squeeze=False)
for ax, variant in zip(axes.ravel(), variants):
    t = placebo.xs(variant, level="variant").reindex(labels_shown)
    y = np.arange(len(labels_shown))
    ax.hlines(y, t["Placebo 5th pct"], t["Placebo 95th pct"], color="lightgrey", lw=6, label="placebo 5th-95th pct")
    ax.plot(t["Placebo median"], y, "|", color="grey", ms=12, label="placebo median")
    ax.plot(t["Actual Sharpe"], y, "o", color="crimson", label="actual")
    ax.set_yticks(y)
    ax.set_yticklabels(labels_shown, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(f"{variant} shift")
    ax.set_xlabel("Sharpe ratio, net of costs")
axes.ravel()[0].legend(fontsize=8, loc="lower right")
plt.tight_layout()
fig.savefig(FIG / "placebo_sharpe.png", dpi=120)
plt.show()
placebo.round(3)

**Deflated Sharpe ratio.** 25 trials: the 10 current ones (9 main specifications and the seed average) from this run, the 15 historical ones from the frozen `dsr_trials.csv` of the reference run. A DSR above 0.95 is significant at 5%.

In [ ]:
trials = stats.dsr_trials(performance, names, main_models, cfg)
dsr = stats.dsr_table(results, market.rf, list(names.values()) + ["Always-on (vol-scaled)"], trials)
print(f"{trials.notna().sum()} trials | SR0 = {dsr['SR0 (annualised)'].iloc[0]:.3f} (annualised)")
dsr.round(3)

**Diebold–Mariano.** Brier loss averaged across ETFs each day, one-sided (does the LSTM have lower loss?), with Newey–West variance over horizon − 1 lags and the Harvey–Leybourne–Newbold correction.

In [ ]:
diebold_mariano = stats.diebold_mariano_tests(stats.forecast_losses(result.probabilities, labels), names, cfg)
diebold_mariano.round(4)

## 10. LSTM across seeds

One row per seed, their median and range, the number of seeds below 5% or with DSR above 0.95, and the seed-averaged forecast as a robustness check next to them.

In [ ]:
lstm_seeds = evaluate.lstm_seed_summary(names, prediction, performance, edges, dsr, placebo, sharpe_tests, variants)
lstm_seeds.round(3)

## 11. Sub-periods

GFC, COVID, the 2022 rate shock, and either side of the 2018 break; dates fixed in `config.yaml` before the results were computed.

In [ ]:
sub_names = ["SPY buy-and-hold", "Vol-targeted SPY", "Always-on (vol-scaled)"] + list(names.values())
subperiods = evaluate.subperiod_metrics({n: results[n] for n in sub_names}, market.rf, result.probabilities, labels, names, cfg)
seed_names = [n for c, n in names.items() if c.startswith("lstm_") and c != "lstm_avg"]
subperiod_seeds = evaluate.subperiod_seed_summary(subperiods, seed_names)
subperiods.pivot(index="period", columns="strategy", values="Sharpe").reindex(
    index=list(dict.fromkeys(subperiods["period"])), columns=sub_names).round(2)

In [ ]:
subperiod_seeds.round(3)

## 12. Save the run

Small tables at the top of the run folder, figures in `figures/`, predictions, equity curves and placebo draws in `large/` (git-ignored), and `run_info.json`.

In [ ]:
evaluate.save_tables(
    out,
    {
        "prediction_metrics": prediction, "auc_by_year": auc_year, "calibration": calibration,
        "tuning_by_window": result.tuning, "tuning_edges": tuning_edges, "signal_edges": edges,
        "portfolio_performance": performance, "cost_sensitivity": costs,
        "feature_selection_stability": frequency, "selection_by_family": families, "selection_stability_summary": stability,
        "sharpe_bootstrap_tests": sharpe_tests, "placebo_tests": placebo, "dsr_trials": trials.to_frame(),
        "deflated_sharpe": dsr, "diebold_mariano": diebold_mariano, "lstm_seeds": lstm_seeds,
        "subperiods": subperiods, "subperiods_lstm_seeds": subperiod_seeds,
    },
    large={
        "predictions": result.probabilities.join(labels.rename("label")),
        "equity_curves": pd.DataFrame({n: r["ret"] for n, r in results.items()}),
        "placebo_draws": placebo_draws,
    },
)
info = evaluate.write_run_info(
    out, cfg, clock,
    label=RUN_LABEL,
    data_source="synthetic" if cfg["run"]["quick_mode"] else "cache",
    quick_mode=cfg["run"]["quick_mode"],
    data_dates=[str(prices.index[0].date()), str(prices.index[-1].date())],
    backtest_dates=[str(common[0].date()), str(common[-1].date())],
    test_windows=[s.test_year for s in folds],
    lstm_seeds=[int(c.split("_")[1]) for c in names if c.startswith("lstm_") and c != "lstm_avg"],
    deflated_sharpe={"trials_used": int(trials.notna().sum()), "sr0_annualised": float(dsr["SR0 (annualised)"].iloc[0])},
    placebo_draws_used=len(placebo_draws),
)
print(f"Saved to {out} | {info['runtime_minutes']} min wall, {info['cpu_minutes']} CPU min")

## Findings on market data

Run `results/2026-10-31_4c11a4d_src-full/` (cached data, test years 2008–2026, net of 5 bps per side). The LSTM is reported as the median and range across 10 seeds; the seed-averaged forecast is shown next to it as a robustness check, never instead of it.

**Reproduction (Step 8).** This `src/` run reproduces the prototype reference run `results/2026-10-30_363ec0b_lstm-10seeds-dsr/` within the stated tolerances. All 13 forecast columns (linear, LightGBM, 10 LSTM seeds, seed average) are identical: max abs diff 0 (tolerance 1e-9 for linear and LightGBM; AUC within 0.001 and Sharpe within 0.01 for each LSTM seed). In the tables built on them, linear, LightGBM and benchmark rows are identical (0), except the deflated Sharpe ratio (9e-9, through SR0, which uses the LSTM trials) and the prediction metrics (5e-5, because the reference rounded them to 4 decimals). LSTM rows differ by at most 5e-5, because `src/` computes signals in float64 where the reference used float32.

**No model adds reliable value net of costs over the always-on portfolio.** Always-on earns a Sharpe ratio of 0.58. The LSTM's median is 0.61 (range 0.49–0.70), and no seed beats always-on significantly: bootstrap median p = 0.41, Ledoit–Wolf median p = 0.44 (range 0.24–0.74), 0 of 10 seeds below 0.05 on either test. The seed average (robustness check) earns 0.67, with p = 0.25 (bootstrap) and 0.28 (Ledoit–Wolf). LightGBM (0.26) and the linear model (0.18) fall below always-on; for the linear model the shortfall is significant, with the 95% bootstrap interval of the difference entirely below zero (−0.74 to −0.04).

**The LSTM has a small but consistent ranking edge (AUC above 0.5 for every seed, median 0.509, range 0.504–0.517).** LightGBM's AUC is 0.506 and the linear model's 0.496. The exposure-matched placebo tests timing, not ranking: it keeps each ETF's exposure and holding lengths and asks whether the actual timing beats randomly shifted timing. The LSTM's median placebo p is 0.075 (independent shift) and 0.095 (common shift), with 3 of 10 seeds below 0.05 in each variant (seed average, robustness check: 0.015 and 0.040). LightGBM (0.80, 0.44) and the linear model (0.99, 0.82) show no timing ability. After deflating for 25 trials (SR0 = 0.33), the LSTM's median deflated Sharpe ratio is 0.87 (range 0.74–0.94), with no seed above 0.95 (seed average, robustness check: 0.92).

**The edge is in ranking, not in the probability levels.** The probability levels are less accurate than a constant base-rate forecast: every model's Brier score is above the base-rate Brier of 0.2451 (LSTM seeds 0.2464–0.2495). No seed has significantly lower forecast loss than LightGBM in the Diebold–Mariano test (0 of 10), and the calibration table shows the LSTM too confident at both ends of its range.

**H4 is not rejected against LightGBM.** LSTM vs LightGBM, bootstrap with Holm correction: median p = 0.077 across seeds (range 0.039–0.177; 3 of 10 below 0.05). Ledoit–Wolf finds no seed significant even before the Holm correction (median p = 0.094, range 0.055–0.199; 0 of 10 below 0.05). The seed average (robustness check) gives p = 0.040 on the bootstrap but 0.063 on Ledoit–Wolf, so it is not robust to the choice of test. The LSTM's significant lead over the linear model (8 of 10 seeds below 0.05 after Holm; 10 of 10 on Ledoit–Wolf) reflects the linear model finding no usable signal, not LSTM skill, and is not robust to the ridge penalty grid: ridge picks its strongest penalty in 7 of 19 windows, and the linear model stays flat in 7 windows.

**Lower drawdowns come from low exposure, not timing.** The LSTM is invested about 27% of the time (seed median), against 75% for always-on. In the GFC its median drawdown is 5%, against 20% for always-on and 46% for SPY, with the median seed invested 8% of the time. At the same exposure, randomly shifted signals do as well or better: placebo median drawdown 9%, against the LSTM's median 11%.

**Costs.** The LSTM's median break-even cost is 47 bps per side (range 43–61), against 937 bps for always-on, which rarely trades; the 5 bps base case is well inside both.

**Before and after the September 2018 break (descriptive, not tested).** The LSTM's median Sharpe ratio is 0.69 before and 0.51 after, against 0.71 and 0.43 for always-on and 0.53 and 0.65 for SPY.